# **Purpose of Notebook**
This notebook assesses the precision of *DetectOffice* function.

In [79]:
import sys, json, os, cv2
import pandas as pd

Year,Showa='1941','16'
Quality='HQ'
origin='C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)'
df = pd.read_csv(origin+'\Tokyo_Jobs\Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

In [80]:
from google.cloud import vision
from google.cloud.vision_v1 import types
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = origin+'\\Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'

In [81]:
origin+'\\Tokyo_Jobs\\Processed_Data\\Data_Collection\\Extract_Data'

'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Data_Collection\\Extract_Data'

# Experiment

In [82]:
StepAError,StepBError,StepCError,MainError=[],[],[],[]
Dict={}
for Page in range(15, 125, 10):    
    file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.jpg'
    img=cv2.imread(os.path.join(origin,file_path))
    
    #Get Office Info: StepA
    try:        
        BookPage=2*Page-14
        Rows=df[(df['Page']==BookPage)]
        NxRows=df[(df['Page']==BookPage+1)]
        if len(Rows)==0:
            print('No Office at Right Side. Page'+str(BookPage))
        if len(NxRows)==0:
            print('No Office at Left Side. Page'+str(BookPage+1))
    except:
        StepAError.append(Page)
        continue
        
    #Apply OCR: StepB
    try:        
        sys.path.append(origin+'\\Tokyo_Jobs\\Data_Collection\\Extract_Data')
        from Read import Vision
        # Instantiates a client
        client = vision.ImageAnnotatorClient()
        texts=Vision(img,'zh',client)
    except:
        StepBError.append(Page)
        continue
        
    #Filter Data: StepC
    try:
        sys.path.append(origin+'\\Tokyo_Jobs\\Data_Collection\\Split_Page')
        from Split import VertPart
        sys.path.append(origin+'\\Tokyo_Jobs\\Data_Collection\\General')
        from PageCut import HoriPart
        from Organize import Filter, ConvertDict
        file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.jpg'
        path=os.path.join(origin,file_path)

        HoriPoint=HoriPart(img,Page,texts)[0]

        try:
            VertPoint=VertPart(path)[1]
        except:
            print('Failed detecting Vertical Point')
            VertPoint=1300

        ##Right Page
        BoxR=Filter(BookPage,texts,HoriPoint)
        BoxR=ConvertDict(BoxR)

        ##Left Page
        BoxL=Filter(BookPage+1,texts,HoriPoint)
        BoxL=ConvertDict(BoxL)

    except:
        StepCError.append(Page)
        continue

        
    
    #Get Office Location: MainStep
    #from Detect import DetectOffice
    from Organize import FilterBox
    from Detect import DetectOffice
    LocList=[]
    try:        
        #RightPage
        OfficeList=Rows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxR, Office,VertPoint))
            BoxR=FilterBox(BoxR,LocList,VertPoint)

        #LeftPage
        OfficeList=NxRows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxL, Office,VertPoint))
            BoxL=FilterBox(BoxL,LocList,VertPoint)

        Dict[Page]=LocList
    except:
        MainError.append(Page)
        continue

Horizontal Line was not automatically detected... Defining line arbitrariry...
Horizontal Line was not automatically detected... Defining line arbitrariry...
No Office at Left Side. Page117
No Office at Left Side. Page157
No Office at Right Side. Page176
No Office at Right Side. Page196
No Office at Left Side. Page217


# Summary of performance

**1. Non-Running Error**

In [83]:
from Show import Show
def ErrorRate(ErrorList,PageList):
    return(round(len(ErrorList)/len(range(10, 120, 10)),2))

In [84]:
ErrorRate(StepAError,range(10, 120, 5)),ErrorRate(StepBError,range(10, 120, 5)),ErrorRate(StepCError,range(10, 120, 5)),ErrorRate(MainError,range(10, 120, 5))

(0.0, 0.0, 0.09, 0.0)

In [87]:
StepCError

[15]

**Results** 

Causes of Main Error
- Cannot find words (Old japanese letter) (0).
- Word not found (1).

**Results**: 5% of pages did not go through

**2.Miss Assignment Error**

In [86]:
for Page in list(Dict.keys()):
    file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.jpg'
    img=cv2.imread(os.path.join(origin,file_path))
    LocList=Dict[Page]
    if len(LocList)!=0:
        for n in range(len(LocList)):
            print(Page,LocList[n]['OfficeName'],LocList[n]['Words'])
        Show(img,LocList)

25 青年教育課 青年教育課八
25 社會教育課 社會教育課主事
35 作業課 作業課主事
35 計畫課 計畫課
45 管理課 管理課課
45 醫務課 醫務
45 築地病院 築地病院京橋區築
55 本郷區出張所 本鄉區出張所(
55 下谷區出張所 下谷區出張所
55 浅草區出張所 草區出張所(
55 本所區出張所 本所區出張所
55 深川區出張所 深川區出張所
55 品川區出張所 品川區出張所
55 目黒區出張所 目黑區出張所(大崎
55 荏原區出張所 荏原區出張所(荏⑧
55 大森區出張所 大森區出張所(大森
55 蒲田區出張所 蒲田區出張所(蒲田二
55 世田谷區出張所 世田谷區出張所(世田谷三
65 給水課 給水課
75 目黒電車營業所 目黑電車營業所(大型
75 廣尾電車營業所 廣尾電車營業所(高
75 青山電車營業所 青山電車營業所(青〇〇
75 新宿電車營業所 新宿電車營業所(5〇七
85 病院 病技師
95 浅草區役所 草區役所正
105 淀橋區役所 淀橋區役所區長主事
115 向島區役所 向島區役所


- Cause of Errors
    1. Reading Names of office at right/left top corners: (0)

    2. Confused with names: (0)
    
        <= Precision should increase by better line detection.
        
    3. Searching in areas of previous office: (0)
        
    4. Wrong Office Name: (0)
        
    5. Wrong Office Page: (0)
    
    6. Word not found in page: (1)